In [ ]:
# ruff: noqa: ANN001, PTH123

# ############################################################################
# MODULE:      start_process
# AUTHOR(S):   Jonas Pischke, Victoria-Leandra Brunn, Julia Haas
#
# PURPOSE:     wrapper script to set variables and execute processing
#
# SPDX-FileCopyrightText: (c) 2025 by mundialis GmbH & Co. KG
#
# SPDX-License-Identifier: GPL-3.0-or-later
#
############################################################################

In [ ]:
import json
import time
from pathlib import Path

import requests
from eodag import EODataAccessGateway
from requests.auth import HTTPBasicAuth


Define variables:

In [12]:
GRASS_PROJECT = "athen_urban-green_epsg32635"
START = "2025-12-01"
END = "2025-12-03"
TILE_ID = "35SKC"
MAIN_PC_PATH = "templates/"
MAIN_PROCESS_CHAIN = "pc_MAIN.json"

Define actinia variables:

In [13]:
# variables to set the actinia host, version, and user
actinia_baseurl = "http://localhost:8088"
actinia_version = "v3"
actinia_url = actinia_baseurl + "/api/" + actinia_version
actinia_auth = HTTPBasicAuth("actinia", "actinia")  # username, password

### Filter Sentinel-2 scenes

In [14]:
search_criteria = {
    "productType": "S2MSI2A",
    "start": START,
    "end": END,
    "tileIdentifier": TILE_ID,
}

In [15]:
dag = EODataAccessGateway()
all_products = dag.search_all(**search_criteria)
s2_ids = [all_products[i].properties["id"] for i in range(len(all_products))]
print(f"Found <{len(s2_ids)}> Sentinel-2 scene IDs:")
for sid in s2_ids:
    print(f" - {sid}")

Found <1> Sentinel-2 scene IDs:
 - S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209


### Start actinia

In [16]:
# helper function to print formatted JSON using the json module
class HasBeenTerminatedError(Exception):
    """Throw exception class."""

    def __init__(self, request_url) -> None:
        """Throw exception."""
        super().__init__(f"The resource <{request_url}> has been terminated.")


def print_as_json(data) -> None:
    """Print request as json."""
    print(json.dumps(data, indent=2))


# helper function to verify a request
def verify_request(request, success_code=200) -> None:
    """Verify the request."""
    if request.status_code != success_code:
        print(
            "ERROR: actinia processing failed with status code "
            f"{request.status_code}!",
        )
        print("See errors below:")
        print_as_json(request.json())
        request_url = request.json()["urls"]["status"]
        requests.delete(url=request_url, auth=actinia_auth, timeout=20)
        raise HasBeenTerminatedError(request_url)

Construct actinia endpoint:
`...locations/{GRASS_PROJECT}/mapsets/PERMANENT/processing` -> persistent, data is kept, needs
`...locations/{GRASS_PROJECT}/processing_export` -> non-persistent, data is not kept in GRASSDB

In [17]:
# make a POST request to the actinia data API
request_url = f"{actinia_url}/locations/{GRASS_PROJECT}/processing_export"

print("actinia POST request:")
print(request_url)
print("---")

actinia POST request:
http://localhost:8088/api/v3/locations/athen_urban-green_epsg32635/processing_export
---


In [18]:
# loop over all Sentinel-2 scene IDs and start processing
# list for storing status request URLs
request_url_lst = []
for s2_id in s2_ids:
    print(f"Processing Sentinel-2 scene ID: {s2_id}")
    # load the main process chain
    with open(MAIN_PC_PATH + MAIN_PROCESS_CHAIN, encoding="utf-8") as file:
        process_chain = json.load(file)
    # modify the process chain with the current Sentinel-2 scene ID
    process_chain["list"][0]["inputs"][0]["value"] = s2_id
    # modify the output names for NDVI and NDWI
    process_chain["list"][5]["inputs"][2][
        "value"
    ] = f"NDVI_{s2_id.split('_')[2][:8]}_{s2_id.split('_')[0]}"
    process_chain["list"][6]["inputs"][2][
        "value"
    ] = f"NDWI_{s2_id.split('_')[2][:8]}_{s2_id.split('_')[0]}"

    # make the POST request to start the processing
    request = requests.post(
        url=request_url, auth=actinia_auth, json=process_chain, timeout=20,
    )
    # check if anything went wrong
    verify_request(request, 200)
    # get a json-encoded content of the response
    json_response = request.json()
    # save the status request URL
    request_url_lst.append(json_response["urls"]["status"])

    while json_response["status"] in {"accepted", "running"}:
        status_request_url = json_response["urls"]["status"]
        status_request = requests.get(
            url=status_request_url.replace("https", "http"), auth=actinia_auth,
            timeout=20,
        )
        json_response = status_request.json()
        print(f"Polling status for {s2_id}: {json_response['status']}")
        print(f"Doing: {json_response['message']}")
        print("#########################################")
        time.sleep(30)

    print(f"Final status for {s2_id}: {json_response['status']}")
    print(f"Processing for {s2_id} finished.")
    print("=========================================")

Processing Sentinel-2 scene ID: S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209


ConnectionError: HTTPConnectionPool(host='localhost', port=8088): Max retries exceeded with url: /api/v3/locations/athen_urban-green_epsg32635/processing_export (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f6bc4393c70>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [ ]:
json_response

{'accept_datetime': '2025-12-15 11:12:20.886874',
 'accept_timestamp': 1765797140.886868,
 'api_info': {'endpoint': 'gdiasyncephemeralexportresource',
  'method': 'POST',
  'path': '/api/v3/locations/athen_urban-green_epsg32635/processing_export',
  'request_url': 'http://localhost:8088/api/v3/locations/athen_urban-green_epsg32635/processing_export'},
 'datetime': '2025-12-15 11:17:19.854431',
 'http_code': 200,
 'message': 'Processing successfully finished',
 'process_chain_list': [{'list': [{'id': 'i.s2_id.download_1',
     'inputs': [{'param': 's2_id',
       'value': 'S2A_MSIL2A_20251202T091401_N0511_R050_T35SKC_20251202T122209'},
      {'param': 'download_dir', 'value': '/home/data/s2'}],
     'module': 'i.s2_id.download'},
    {'flags': 'cr',
     'id': 'i.sentinel.import_1',
     'inputs': [{'param': 'input', 'value': '/home/data/s2'},
      {'param': 'pattern', 'value': 'B(02_1|03_1|04_1|08_1|8A_2|11_2|12_2)0m'},
      {'param': 'cloud_probability_threshold', 'value': '65'},
  

Print links to tif of NDVI/NDWI results

In [ ]:
# print links to processed tifs
for s2_id in request_url_lst:
    status_request = requests.get(
        url=s2_id.replace("https", "http"), auth=actinia_auth, timeout=20,
    )
    json_response = status_request.json()
    for tif in json_response["urls"]["resources"]:
        print(f"{Path(tif).name}: {tif.replace('https', 'http')}")

NDVI_20251202_S2A.tif: http://localhost:8088/api/v3/resources/actinia/resource_id-953b2fe0-c6d9-4d3f-9dde-287099ce9436/NDVI_20251202_S2A.tif
NDWI_20251202_S2A.tif: http://localhost:8088/api/v3/resources/actinia/resource_id-953b2fe0-c6d9-4d3f-9dde-287099ce9436/NDWI_20251202_S2A.tif
